[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/stress/stress.ipynb)

# Stress in an Elastic Cube

This example computes the Cauchy stress on the faces of a strained elastic cube and follows its normal and shear tractions across reference frames. Every cut face of a deformed block experiences a force per unit area composed of a normal pull or push across the face and a tangential shear along it. Rotating the cut redistributes those two components without changing the material state: aligned with the principal axes, every face is free of shear, whereas turning the cut traces Mohr's circle up to the plane of greatest shear.

Strain is modelled as a map `Plane <- Point` from material points to the planes that read their stretch, while Cauchy stress is a map `Point <- Plane` sending each face's plane to the traction across that face. In an isotropic solid, dualizing `Plane` supplies the unit normal direction across a face, `strain(Plane.dual()).trace()` reads the volume dilatation, and `lame * dilatation * Plane.dual() + 2 * shear_modulus * strain(Plane.dual()).dual()` gives Hooke's law directly. Pairing the stress with a second plane builds the symmetric form `Plane & stress(Plane)`, whose generalized eigenproblem `(Plane & stress(Plane)).eigh(Plane | Plane)` yields the principal stresses and planes, and rotating the reference frame transforms the stress by `rotor << stress(rotor >> Plane)`.

In Cartesian tensor notation, isotropic Hooke's law introduces coordinate indices and a Kronecker delta, $\sigma_{ij} = \lambda\,\varepsilon_{kk}\,\delta_{ij} + 2\mu\,\varepsilon_{ij}$, Cauchy's traction formula contracts with the unit normal components $t_i = \sigma_{ij} n_j$, and rotating into a frame with orthogonal matrix $R_{ip}$ transforms the stress components by $\sigma'_{pq} = R_{ip} R_{jq} \sigma_{ij}$ before splitting normal traction $\sigma_n = n_i \sigma_{ij} n_j$ from tangential shear $s_i = t_i - \sigma_n n_i$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D
from examples.util.animation import save_figure, show_states
from examples.mechanics.stress import render, scenarios

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                # Scalar+
# Points and displacements are intrinsic antivectors; cut faces are extrinsic planes.
Point = ga.gatype.antivector()                             # Antivector-
Plane = ga.gatype.vector()                                 # Vector+
Bivector = ga.gatype.bivector()                            # Bivector+
Rotor = ga.gatype.rotor()                                  # Rotor+
Strain = ga.gatype((Plane, Point))                         # Plane+ <- Point-
Stress = ga.gatype((Point, Plane))                         # Point- <- Plane+

axes = mv.basis()                                          # [3] Plane+
x, y, z = axes
# Unit cube centred on the origin: eight corners and six outward-facing planes.
corners = mv.vector(np.stack(np.meshgrid(*[[-0.5, 0.5]] * 3, indexing="ij"), axis=-1).reshape(-1, 3)).dual()   # [8] Point-
faces = concatenate([axes, -axes])                         # [6] Plane+

## Isotropic strain and Cauchy stress

A linear strain maps each material point to the plane that reads its stretch, and dualizing that plane gives the point's displacement. Summing weighted dyads `axes * (axes & Point)` imposes triaxial stretches along the coordinate axes while symmetric cross terms shear the cube in the `xy` plane. Isotropic Hooke's law combines the volumetric dilatation from the trace `strain(Plane.dual()).trace()` along the face normal `Plane.dual()` with twice the shear modulus along the dualized strain reading `strain(Plane.dual()).dual()`, and pairing a face's traction with its plane separates the normal pull across the face from the tangential shear.

In [ ]:
lame = scenarios.lame
shear_modulus = scenarios.shear_modulus
stretches = scenarios.stretches
shear_strain = scenarios.shear_strain

In [ ]:
def cauchy_stress(strain: Strain, lame: float, shear_modulus: float) -> Stress:
    """Isotropic Hooke's law maps `strain` to the Cauchy stress (`[] Point- <- Plane+`) with first
    Lamé parameter `lame` and shear modulus `shear_modulus`."""
    # Trace of the strain read along the metric duals of the basis planes gives the volume dilatation.
    dilatation = strain(Plane.dual()).trace()                                  # [] Scalar+
    # Dualizing `Plane` gives the unit normal direction across a face, and dualizing `strain(...)` turns its plane reading back into a traction direction.
    return lame * dilatation * Plane.dual() + 2 * shear_modulus * strain(Plane.dual()).dual()   # [] Point- <- Plane+


def face_tractions(stress: Stress, cut_faces: Plane) -> tuple[Point, Point]:
    """Each unit plane in `cut_faces` (`[...] Plane+`) receives a traction under `stress` that splits
    into its normal component (`[...] Point-`) across the face and its shear component (`[...] Point-`)
    along the face."""
    total = stress(cut_faces)                                                  # [...] Point-
    normal = cut_faces.dual() * (cut_faces & total)                            # [...] Point-
    return normal, total - normal


strain = (axes * stretches * (axes & Point)).sum(axis=0) + shear_strain * (x * (y & Point) + y * (x & Point))   # [] Plane+ <- Point-
stress = cauchy_stress(strain, lame, shear_modulus)                            # [] Point- <- Plane+

## Principal axes and Mohr's circle

Pairing `stress(Plane)` with a second `Plane` forms a symmetric bilinear form whose generalized eigenproblem against `Plane | Plane` yields the three ordered principal stresses and principal planes. Because the imposed shear lies in `xy`, the third axis stays principal and the rotor `(1 + planes[0] * ~x).normalized()` aligns the reference frame with the principal planes. Half the sum and half the difference of the extreme principal stresses set the centre and maximum-shear radius of the outer Mohr circle.

In [ ]:
def principal_frame(stress: Stress) -> tuple[Scalar, Point, Rotor]:
    """Solving the symmetric eigenproblem of `Plane & stress(Plane)` against `Plane | Plane` gives
    the ordered principal stresses (`[3] Scalar+`), the principal directions (`[3] Point-`), and the
    rotor (`[] Rotor+`) carrying the first basis plane onto the first principal plane."""
    # Pairing `stress(Plane)` with `Plane` forms a symmetric bilinear form whose eigenvectors against `Plane | Plane` are the principal planes.
    stresses, planes = (Plane & stress(Plane)).eigh(Plane | Plane)             # [3] Scalar+, [3] Plane+
    return stresses, planes.dual(), (1 + planes[0] * ~axes[0]).normalized()


def mohr_circle(stresses: Scalar) -> tuple[Scalar, Scalar]:
    """The least and greatest of the ordered principal `stresses` (`[3] Scalar+`) set the normal-stress
    centre (`[] Scalar+`) and maximum-shear radius (`[] Scalar+`) of the outer Mohr circle."""
    return (stresses[0] + stresses[2]) / 2, (stresses[2] - stresses[0]).abs() / 2


stresses, directions, principal = principal_frame(stress)                      # [3] Scalar+, [3] Point-, [] Rotor+
centre, radius = mohr_circle(stresses)                                         # [] Scalar+, [] Scalar+

## Comparing reference frames

Seen from a rotated frame, the strain and stress maps conjugate on both input and output as `rotors << strain(rotors >> Point)` and `rotors << stress(rotors >> Plane)`. Evaluating the lab frame, the principal frame, and a frame turned an eighth of a turn further in `xy` places pure normal traction beside maximum shear.

In [ ]:
def frame_views(strain: Strain, stress: Stress, directions: Point, rotors: Rotor
                ) -> tuple[Point, Point, Point, Point, Point]:
    """Transforming `strain`, `stress`, and the principal `directions` (`[3] Point-`) into each frame
    of `rotors` (`[...] Rotor+`) gives the deformed cube corners (`[..., 8] Point-`), face centres
    (`[..., 6] Point-`), normal and shear face tractions (`[..., 6] Point-`), and local principal
    directions (`[..., 3] Point-`)."""
    local_strain = rotors << strain(rotors >> Point)                           # [...] Plane+ <- Point-
    local_stress = rotors << stress(rotors >> Plane)                           # [...] Point- <- Plane+
    normal, shear = face_tractions(local_stress[..., None], faces)             # [..., 6] Point-, [..., 6] Point-
    # Dualizing each point's strain plane gives its displacement in the local frame.
    return (corners + local_strain[..., None](corners).dual(),                 # [..., 8] Point-
            0.5 * (faces.dual() + local_strain[..., None](faces.dual()).dual()),   # [..., 6] Point-
            normal, shear, rotors[..., None] << directions)                    # [..., 3] Point-


eighth_turn = (mv.xy * (-np.pi / 8)).exp() * principal                         # [] Rotor+
rotors = stack([mv.rotor(), principal, eighth_turn])                           # [3] Rotor+
views = frame_views(strain, stress, directions, rotors)
save_figure(render.frames_figure(views, stresses), "stress_frames");

The first three panels show the deformed translucent cube inside the pale undeformed wireframe in the lab frame, the principal frame, and the maximum-shear frame, with blue normal tractions, red shear tractions, and green principal axes. The fourth panel plots Mohr's circles with normal traction horizontal and shear vertical, where the blue, green, and red diameters connect the `+x` face (filled dot) and `+y` face (open ring) of each frame.

## Sweeping a half-turn through the shear plane

Turning the reference frame through half a turn in `xy` from the principal frame carries the cube through maximum shear and back to a shear-free alignment while its diameter sweeps a full revolution around Mohr's circle.

In [ ]:
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def turning_views(strain: Strain, stress: Stress, directions: Point, principal: Rotor, plane: Bivector,
                  n_frames: int) -> Iterator[tuple[Point, Point, Point, Point, Point]]:
    """Yields `frame_views` across `n_frames` evenly spaced angles over half a turn in `plane`
    (`[] Bivector+`) starting from `principal`."""
    for angle in np.linspace(0.0, np.pi, n_frames, endpoint=False):
        rotor = (plane * (-angle / 2)).exp() * principal                       # [] Rotor+
        yield frame_views(strain, stress, directions, rotor)


swept = show_states(turning_views(strain, stress, directions, principal, mv.xy, n_frames),
                    render.turning_frames, stresses, name="stress_rotation", duration_ms=frame_duration_ms,
                    n_frames=n_frames);

In [ ]:
# checks
# In the principal frame no face is sheared, the normal tractions match the principal stresses, and stress acts purely along the principal normals.
_, _, seen_normal, seen_shear, _ = views
assert (seen_shear[1].norm() < 1e-12).all()
principal_normals = axes & seen_normal[1, :3]                                  # [3] Scalar+
assert ((principal_normals[principal_normals.argsort()] - stresses).abs() < 1e-12).all()
principal_planes = principal >> axes                                           # [3] Plane+
assert ((principal_planes.dual() & stress(principal_planes)).norm() < 1e-12).all()
# Turning an eighth of a turn from the principal frame reaches the outer Mohr circle's maximum-shear radius on the +x face.
assert ((seen_shear[2, 0].norm() - radius).abs() < 1e-8).all()
# Across the streamed rotation, the +x face's normal and shear tractions lie on Mohr's circle.
swept_normal = stack([state[2][0] for state in swept])                         # [n_frames] Point-
swept_shear = stack([state[3][0] for state in swept])                          # [n_frames] Point-
normal_component, shear_component = x & swept_normal, y & swept_shear          # [n_frames] Scalar+, [n_frames] Scalar+
circle_residual = (normal_component - centre).squared() + shear_component.squared() - radius.squared()   # [n_frames] Scalar+
assert (circle_residual.abs() < 1e-8).all()